# Deterministic Biomethane SMR hydrogen NPV

Calculate expected-input NPV, levelized profit margin, and LCOH at 100,000 tH2/year. The source model uses a 25-year lifetime and 7,500 EUR/tH2 retail price. Uncertain inputs use their analytical means. Biomethane SMR is a fuel-switch retrofit to NG-SMR without CCS: natural gas is replaced by biomethane at a fixed price of 87.5 EUR/MWh_th. The parent and retrofit inputs are shown separately below.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hydrogen.hydrogen_npv_deterministic import calculate_deterministic_hydrogen_result
from hydrogen.hydrogen_financial_summary import (
    HYDROGEN_PROCESSED_OUTPUT_COLUMNS,
    HYDROGEN_RAW_INPUT_COLUMNS,
    HYDROGEN_TECHNOLOGY_LABELS,
)

pd.options.display.float_format = "{:,.3f}".format

In [ ]:
TECHNOLOGY = "biomethane_smr"
result = calculate_deterministic_hydrogen_result(TECHNOLOGY)
values = {key: item[0] for key, item in result.items()}

UNITS = {
    "annual_output_th2": "tH2/year",
    "lifetime_years": "years",
    "capex_eur_per_th2": "EUR/(tH2/y)",
    "fixed_opex_eur_per_th2": "EUR/tH2",
    "variable_opex_eur_per_th2": "EUR/tH2",
    "natural_gas_consumption_mwh_per_th2": "MWh/tH2",
    "biomethane_consumption_mwh_per_th2": "MWh/tH2",
    "biomass_consumption_mwh_per_th2": "MWh/tH2",
    "electricity_consumption_mwh_per_th2": "MWh/tH2",
    "emissions_tco2_per_th2": "tCO2/tH2",
    "gas_price_eur_per_mwh_th": "EUR/MWh_th",
    "biomethane_price_eur_per_mwh_th": "EUR/MWh_th",
    "biomass_price_eur_per_mwh_th": "EUR/MWh_th",
    "electricity_price_eur_per_mwh": "EUR/MWh",
    "hydrogen_price_eur_per_th2": "EUR/tH2",
    "carbon_price_eur_per_t": "EUR/tCO2",
    "transport_and_storage_share_of_capture_cost": "fraction",
    "transport_and_storage_cost_eur_per_th2": "EUR/tH2",
    "capture_cost_excluding_transport_and_storage_eur_per_th2": "EUR/tH2",
    "discounted_lifetime_output_th2": "discounted tH2",
    "lcoh_eur_per_th2": "EUR/tH2",
    "levelized_profit_margin_eur_per_th2": "EUR/tH2",
    "capture_fraction": "fraction",
}

def unit_for(key):
    key = key.removeprefix("bau_").replace("_change_", "_")
    if key.endswith("_fraction"):
        return "fraction"
    if key in UNITS:
        return UNITS[key]
    if key.endswith("_eur"):
        return "EUR"
    if key in {"technology", "technology_type", "retrofit_bau_mode", "fuel_type"}:
        return "category"
    return ""

def as_table(keys):
    return pd.DataFrame(
        {"Input / output": key, "Value": values[key], "Unit": unit_for(key)}
        for key in keys if key in values and key != "run_id"
    )

summary = pd.DataFrame([
    {"Metric": "Technology", "Value": HYDROGEN_TECHNOLOGY_LABELS[TECHNOLOGY], "Unit": ""},
    {"Metric": "Annual hydrogen output", "Value": values["annual_output_th2"], "Unit": "tH2/year"},
    {"Metric": "Direct emissions", "Value": values["emissions_tco2_per_th2"], "Unit": "tCO2/tH2"},
    {"Metric": "NPV", "Value": values["npv_eur"] / 1_000_000, "Unit": "million EUR"},
    {"Metric": "Levelized profit margin", "Value": values["levelized_profit_margin_eur_per_th2"], "Unit": "EUR/tH2"},
    {"Metric": "LCOH", "Value": values["lcoh_eur_per_th2"], "Unit": "EUR/tH2"},
])
raw_inputs = as_table(HYDROGEN_RAW_INPUT_COLUMNS)
processed_outputs = as_table(HYDROGEN_PROCESSED_OUTPUT_COLUMNS)
retrofit_keys = [
    key for key in values
    if key.startswith("bau_") or "_change_" in key or key.endswith("_fraction")
]
if TECHNOLOGY == "biomethane_smr":
    retrofit_keys.append("biomethane_consumption_mwh_per_th2")
retrofit_inputs = as_table(retrofit_keys)

## Summary

In [ ]:
summary

## Expected inputs

In [ ]:
raw_inputs

## Parent and incremental retrofit inputs

In [ ]:
retrofit_inputs

## Processed outputs

In [ ]:
processed_outputs